# Этап 3 · два проекта

Immigration Relocator: 11 конкурентов, 9 критериев, 30 наблюдений. SHIFT: отдельный профиль и кандидаты с обязательной проверкой человеком.

Встроены реальные сохранённые снимки. По умолчанию новых LLM-запросов нет. Старые данные 22.10 и дополнения 04.11.2026 различаются датами источников. Это новый исследовательский ноутбук; старый Colab остаётся историей предыдущего этапа.

In [1]:
#@title 1. Подготовить окружение и распаковать сохранённые данные
import base64, hashlib, io, json, lzma, pathlib, subprocess, sys, zipfile
subprocess.check_call([sys.executable,'-m','pip','install','-q','requests>=2.32,<3','beautifulsoup4>=4.12,<5','fastapi==0.162.0','uvicorn==0.54.0','httpx==0.28.1'])
PACKED="""..."""
data=base64.b64decode(PACKED)
assert hashlib.sha256(data).hexdigest()=="7e8cc246291416c1d953b7b4d66f5a9b83f467df29dff3f820ff91db6e2aa7b7"
WORK=pathlib.Path.cwd()/'stage3_two_cases'
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(lzma.decompress(data))) as archive:
    for name in archive.namelist():
        assert (WORK/name).resolve().is_relative_to(WORK.resolve())
    archive.extractall(WORK)
ROOT=WORK/'stage_3'
sys.path.insert(0,str(WORK)); sys.path.insert(0,str(ROOT/'code'))
from analyst import read, run, verify_inputs
verify_inputs(ROOT)
print('Целостность снимков подтверждена. Новых LLM-вызовов: 0.')


Целостность снимков подтверждена. Новых LLM-вызовов: 0.


In [2]:
#@title 2. Запустить анализ релокации
COUNTRY='Portugal' #@param ['Portugal','Spain']
CONFIG=read(ROOT/'config.json'); CONFIG['country']=COUNTRY
RESULT=run(ROOT,CONFIG,ROOT/'results/current')
print('Конкурентов:',len(CONFIG['competitors']))
print('Критериев:',len(RESULT['criteria']),'наблюдений:',len(RESULT['evidence']),'гипотез:',len(RESULT['recommendations']))
for c in RESULT['coverage']: print(c['name'],c['facts_in_demo'],c['status'],c.get('added_by'))
from IPython.display import HTML, display
display(HTML((ROOT/'results/current/report.html').read_text(encoding='utf-8')))


Конкурентов: 11
Критериев: 9 наблюдений: 30 гипотез: 3
Immigration Relocator 8 limited_snapshot None
Migrun 5 limited_snapshot None
Migran 0 not_researched_for_selected_country None
Relotus 0 not_researched_for_selected_country None
Релокейт Групп 0 not_researched_for_selected_country None
e-residence 0 not_researched_for_selected_country None
DISCUS HOLDINGS PRIVATE CLIENTS 0 not_researched_for_selected_country None
MigrAll Portugal 0 not_researched_for_selected_country None
MoveToCascais 4 limited_snapshot user
PortugalDom 5 limited_snapshot user
MONOLITE 5 limited_snapshot user
OKNO Agency 3 limited_snapshot user


In [3]:
#@title 3. Прогнать тесты движка, другой отрасли и API
import unittest
suite=unittest.defaultTestLoader.discover(str(ROOT/'code'),pattern='test_*.py')
suite.addTests(unittest.defaultTestLoader.discover(str(WORK/'app'),pattern='test_api.py',top_level_dir=str(WORK)))
outcome=unittest.TextTestRunner(verbosity=2,stream=sys.stdout).run(suite)
assert outcome.wasSuccessful()
print('PASS:',outcome.testsRun,'тестов; LLM-вызовов: 0')


test_actual_model_is_not_called_in_report_and_html_is_escaped (test_analyst.AnalystTests.test_actual_model_is_not_called_in_report_and_html_is_escaped) ... ok
test_conflict_is_retained_and_price_ranking_blocked (test_analyst.AnalystTests.test_conflict_is_retained_and_price_ranking_blocked) ... ok
test_forged_quote_company_and_number_are_rejected (test_analyst.AnalystTests.test_forged_quote_company_and_number_are_rejected) ... ok
test_manifest_tamper_is_rejected (test_analyst.AnalystTests.test_manifest_tamper_is_rejected) ... ok
test_missing_competitors_not_misrepresented_as_absent_services (test_analyst.AnalystTests.test_missing_competitors_not_misrepresented_as_absent_services) ... ok
test_service_without_price_is_not_zero_or_absence (test_analyst.AnalystTests.test_service_without_price_is_not_zero_or_absence) ... ok
test_spain_produces_no_portugal_claims (test_analyst.AnalystTests.test_spain_produces_no_portugal_claims) ... ok
test_transport_error_saves_safe_failure_without_secret (t

## Здесь нужен человек

Проверьте профиль SHIFT, рынок и кандидатов. Они не подтверждены автоматически. Retensis и Acuity имеют сохранённые страницы; Pacing — только кандидат из поиска, прямой сбор не удался. Новые URL сначала надо собрать. Проверка в тестах использует искусственное подтверждение только для отрицательных сценариев и не считается вашим согласием.

In [4]:
#@title 4. Проверить профиль и обязательную остановку
from profile_site import draft_profile
from review_case import analyze_case
PROFILE=draft_profile(read(ROOT/'cases/shift/own_source.json'))
print(json.dumps(PROFILE,ensure_ascii=False,indent=2))
print(json.dumps(read(ROOT/'cases/shift/result.json')['candidates'],ensure_ascii=False,indent=2))
SHIFT=analyze_case(ROOT/'cases/shift')
assert SHIFT['status']=='needs_user_review'
print('Нужно ваше действие:',SHIFT['next_action'])


{
  "company_name": "SHIFT",
  "headline": "Give your next video more impact.",
  "summary": "Short video analysis: a specific moment, an explanation and an edit you can test.",
  "market_terms": "AI video analysis short form creators",
  "inference_method": "metadata_headings_and_explicit_keyword_rule_no_llm",
  "source_url": "https://youtube.fghk.ru/landing/?lang=en",
  "source_sha256": "e4672003849610ba2c11d87a9bb7d89af344d3cf67ea01d677092a10c9991ba3",
  "review_status": "pending_user_confirmation",
  "search_queries": [
    "AI video analysis short form creators competitors",
    "AI video analysis short form creators timestamp editing feedback",
    "AI video analysis short form creators pricing alternatives"
  ],
  "needs_user_input": [
    "Подтвердить продукт и аудиторию",
    "Уточнить географию и язык рынка",
    "Подтвердить или дополнить конкурентов"
  ],
  "limits": [
    "Черновик по одной странице; утверждения сайта не проверены на практике.",
    "Профиль и запросы треб

In [5]:
#@title 5. Применить ваши уточнения со стенда (по желанию)
UPLOAD_REVIEW=False #@param {type:'boolean'}
if UPLOAD_REVIEW:
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1,'Загрузите один shift-user-review.json'
    REVIEW=json.loads(next(iter(uploaded.values())).decode('utf-8-sig'))
    SHIFT=analyze_case(ROOT/'cases/shift',REVIEW)
    print(json.dumps(SHIFT,ensure_ascii=False,indent=2))
else:
    print('Ожидается проверка пользователя. На стенде заполните рынок, подтвердите профиль и выберите конкурентов; скачайте файл уточнений.')


Ожидается проверка пользователя. На стенде заполните рынок, подтвердите профиль и выберите конкурентов; скачайте файл уточнений.


In [6]:
#@title 6. Ручное дополнение списка и повторный сбор (по желанию)
RUN_COLLECTION=False #@param {type:'boolean'}
NEW_COMPETITOR_URL='' #@param {type:'string'}
if RUN_COLLECTION:
    assert NEW_COMPETITOR_URL.startswith('https://'),'Введите публичный HTTPS URL'
    from profile_site import capture
    NEW_SOURCE=capture(NEW_COMPETITOR_URL)
    candidates=read(ROOT/'cases/shift/candidate_sources.json')
    NEW_SOURCE.update(role='candidate',discovery_method='user_supplied',added_by='user',confirmation_status='pending_user_confirmation')
    candidates=[s for s in candidates if s['requested_url']!=NEW_COMPETITOR_URL]+[NEW_SOURCE]
    (ROOT/'cases/shift/candidate_sources.json').write_text(json.dumps(candidates,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Собрано. Источник требует подтверждения; обновите список в файле проверки перед сравнением.')
else:
    print('Новые сайты не загружались. Сохранённые снимки не перезаписаны.')


Новые сайты не загружались. Сохранённые снимки не перезаписаны.
